# Atividade 01

In [1]:
# Atividade Prática - Extrator de Dados Fiscais: XML para Excel
# Nome: Eduardo Henrique Da Silva

# Etapa 1: Importações e Padrão PEP 8

import os
import xml.etree.ElementTree as ET

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill


# Etapa 2: Coleta e Varredura de XMLs

os.makedirs("notas_xml", exist_ok=True)

notas_amostra = [
    {
        "num": "1001",
        "data": "2026-10-10",
        "fornecedor": "Tech Solutions LTDA",
        "cnpj": "12.345.678/0001-90",
        "total": "4500.00"
    },
    {
        "num": "1002",
        "data": "2026-10-12",
        "fornecedor": "Cloud Cloud Servicos",
        "cnpj": "98.765.432/0001-11",
        "total": "1280.50"
    },
    {
        "num": "1003",
        "data": "2026-10-15",
        "fornecedor": "Hardware & Redes SA",
        "cnpj": "45.123.890/0001-55",
        "total": "8900.00"
    }
]

# Criando os arquivos XML simulados
for nota in notas_amostra:
    conteudo = f"""<?xml version="1.0" encoding="UTF-8"?>
<NFe>
    <infNFe>
        <ide>
            <nNF>{nota['num']}</nNF>
            <dhEmi>{nota['data']}T14:30:00-03:00</dhEmi>
        </ide>
        <emit>
            <CNPJ>{nota['cnpj']}</CNPJ>
            <xNome>{nota['fornecedor']}</xNome>
        </emit>
        <total>
            <ICMSTot>
                <vNF>{nota['total']}</vNF>
            </ICMSTot>
        </total>
    </infNFe>
</NFe>
"""

    with open(
        f"notas_xml/nfe_{nota['num']}.xml",
        "w",
        encoding="utf-8"
    ) as arquivo:
        arquivo.write(conteudo)

print("Amostras criadas na pasta notas_xml!")


# Etapa 3: Extração de Dados

dados_fiscais = []

for nome_arquivo in os.listdir("notas_xml"):

    if nome_arquivo.endswith(".xml"):

        caminho = os.path.join("notas_xml", nome_arquivo)

        try:
            arvore = ET.parse(caminho)
            raiz = arvore.getroot()

            # Localiza as informações no XML
            numero = raiz.find(".//nNF").text
            data_emissao = raiz.find(".//dhEmi").text
            fornecedor = raiz.find(".//xNome").text
            cnpj = raiz.find(".//CNPJ").text
            valor_total = float(raiz.find(".//vNF").text)

            # Formata a data e o valor
            data_emissao = data_emissao.split("T")[0]
            valor_total = float(f"{valor_total:.2f}")

            dados_fiscais.append([
                numero,
                data_emissao,
                fornecedor,
                cnpj,
                valor_total
            ])

            print(f"Nota fiscal {numero} extraída com sucesso!")

        except FileNotFoundError:
            print(f"Erro: Arquivo {nome_arquivo} não encontrado.")

        except ET.ParseError:
            print(f"Erro: XML inválido no arquivo {nome_arquivo}.")

        except AttributeError:
            print(f"Erro: Dados incompletos no arquivo {nome_arquivo}.")

        except ValueError:
            print(f"Erro: Valor numérico inválido em {nome_arquivo}.")

        except Exception as erro:
            print(f"Erro inesperado: {erro}")


# Etapa 4: Geração da Planilha Excel

planilha = Workbook()
aba = planilha.active
aba.title = "Dados Fiscais"

# Cabeçalho
aba.append([
    "Número da Nota",
    "Data de Emissão",
    "Fornecedor",
    "CNPJ",
    "Valor Total"
])

# Inserção dos dados
for dado in dados_fiscais:
    aba.append(dado)

# Formatação do cabeçalho
for celula in aba[1]:
    celula.font = Font(bold=True)
    celula.fill = PatternFill(
        start_color="ADD8E6",
        end_color="ADD8E6",
        fill_type="solid"
    )

# Formatação dos valores monetários
for linha in range(2, aba.max_row + 1):
    aba.cell(row=linha, column=5).number_format = '"R$" #,##0.00'

# Ajustando largura das colunas
aba.column_dimensions["A"].width = 18
aba.column_dimensions["B"].width = 20
aba.column_dimensions["C"].width = 30
aba.column_dimensions["D"].width = 25
aba.column_dimensions["E"].width = 20

# Salvando a planilha
planilha.save("auditoria_fiscal.xlsx")

print("\nArquivo auditoria_fiscal.xlsx criado com sucesso!")
print("Total de notas fiscais extraídas:", len(dados_fiscais))

Amostras criadas na pasta notas_xml!
Erro: XML inválido no arquivo nfe_1003.xml.
Nota fiscal 1001 extraída com sucesso!
Nota fiscal 1002 extraída com sucesso!

Arquivo auditoria_fiscal.xlsx criado com sucesso!
Total de notas fiscais extraídas: 2
